# Acquirium client reference

This is the reference guide for the acquirium client: the query interface feature by feature, with the internals (`show_query_graph`, `to_sparql`, text resolution) shown along the way. If you are new, start with `quickstart.ipynb`.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `acquirium server --config deployments/WATERTAP/scripts/acquirium.toml`
2. Connect:

In [1]:
from datetime import datetime, timedelta, timezone
from acquirium import Acquirium

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

## Find entities by class
`_class` accepts a URI or a natural-language string. `alias` names the node for later reference.

In [2]:
q = acq.find_entity(_class="Pump", alias="pump")
_ = q.metadata_head()

Metadata First
   10 Rows    
┏━━━━━━━━━━━━┓
┃ pump       ┃
┡━━━━━━━━━━━━┩
│ wbs:intake │
│ wbs:P2     │
│ wbs:P1     │
└────────────┘

In [3]:
q = acq.explore().entity("Pump")
q.metadata()

Pump
str
"""wbs:intake"""
"""wbs:P1"""
"""wbs:P2"""


### How strings become URIs
Every string is resolved server-side by an embedding matcher. `resolve_text` shows what a string resolves to — check it when a query returns something unexpected, and pass exact URIs when correctness matters (the top match is not always the intended one):

In [4]:
acq.client.resolve("salt", kind="class", top_k=3)

[{'uri': 'urn:nawi-water-ontology#Salt-NaCl',
  'kind': 'class',
  'label': 'Salt-NaCl',
  'score': 0.8903464674949646,
  'matched_surface': 'salt na cl',
  'match_stage': 'semantic',
  'related': []},
 {'uri': 'urn:nawi-water-ontology#Constituent-Salt',
  'kind': 'class',
  'label': 'Constituent-Salt',
  'score': 0.7883857488632202,
  'matched_surface': 'constituent salt',
  'match_stage': 'semantic',
  'related': []},
 {'uri': 'urn:nawi-water-ontology#pHAdjuster-Lime',
  'kind': 'class',
  'label': 'Lime',
  'score': 0.757838785648346,
  'matched_surface': 'lime',
  'match_stage': 'semantic',
  'related': []}]

## Follow relationships
`find_related` adds a neighbour reachable within `hops`. `predicates` restricts which edges to follow (`multi_hop_predicates=True` applies them at every hop); `direction="upstream"/"downstream"` walks the S223 piping topology instead. Strings and URIs are both accepted.

In [5]:
q = (
    acq.find_entity(_class="Pump", alias="pump")
       .find_related(_class="Tank", alias="tank", _from="pump", hops=1)
)
q.show_query_graph()
_ = q.metadata_head()

QUERY GRAPH

Nodes:
  0 [pump]  class=http://data.ashrae.org/standard223#Pump
  2 [tank]  class=urn:nawi-water-ontology#Tank

Edges:
  pump --(*, hops=1)--> tank

Data nodes: (none)

Current pointer: tank



           Metadata First 10 Rows            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ tank                         ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:intake │ wbs:ferric-chloride-addition │
└────────────┴──────────────────────────────┘

In [6]:
q = (
    acq.explore().entity("Pump")
    .related("tank")
)
q.metadata()


Pump,tank
str,str
"""wbs:intake""","""wbs:ferric-chloride-addition"""
"""wbs:P2""","""wbs:storage-tank-2"""
"""wbs:P1""","""wbs:storage-tank-2"""
"""wbs:P1""","""wbs:anti-scalant-addition"""


## Attach data nodes
`find_data` adds the observable/actuatable properties of the current node. `find_all_data` does it for every entity in the graph.


In [7]:
q = acq.find_entity(_class="Pump", alias="pump").find_data()
_ = q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-toc-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-mechanical-power         │
│ wbs:P2     │ wbs:P2-mechanical-power         │
│ wbs:P2     │ wbs:P2-efficiency               │
└────────────┴─────────────────────────────────┘

In [8]:
q = acq.explore().entity("Pump").measurement()
q.metadata()

Pump,Pump_data
str,str
"""wbs:P1""","""wbs:P1-out-pressure"""
"""wbs:P1""","""wbs:P1-mechanical-power"""
"""wbs:P2""","""wbs:P2-mechanical-power"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…"
"""wbs:intake""","""wbs:intake-in-tds-concentratio…"
"""wbs:intake""","""wbs:intake-in-flow-rate"""
"""wbs:P1""","""wbs:P1-efficiency"""
"""wbs:P2""","""wbs:P2-efficiency"""
"""wbs:intake""","""wbs:intake-in-toc-concentratio…"


In [9]:
q_all = acq.find_all_data()
_ =q_all.metadata_head()

               Metadata First 10 Rows                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:storage-tank-3-out-flow-rate                  │
│ wbs:RO-out-flow-mass-water                        │
│ wbs:P1-out-pressure                               │
│ wbs:conn-cartridge-filtration-to-S1-pressure      │
│ ns3:point_4                                       │
│ wbs:intake-in-tds-concentration                   │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ ns3:point_1                                       │
│ ns3:point_8                                       │
│ ns3:point_3                                       │
└───────────────────────────────────────────────────┘

In [10]:
acq.explore().measurement().metadata()

data
str
"""wbs:RO-membrane-area"""
"""wbs:RO-out-flow-mass-water"""
"""ns3:point_6"""
"""wbs:P2-efficiency"""
"""ns3:point_9"""
…
"""wbs:conn-cartridge-filtration-…"
"""ns3:point_4"""
"""wbs:storage-tank-3-out-flow-ra…"


## Filter data nodes
Filters apply to the bound data nodes. Strings are resolved via the text matcher.

In [11]:
q = (
    acq.find_all_data()
       .filter_by_quantity_kind("Pressure")
)
_ =q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:PXR-brine-out-pressure                   │
│ wbs:RO-in-pressure                           │
│ wbs:RO-out-retentate-pressure                │
│ wbs:RO-out-pressure                          │
│ wbs:P1-out-pressure                          │
└──────────────────────────────────────────────┘

In [12]:
q = acq.explore().measurement().where(quantity_kind="Pressure")
q.metadata()

data
str
"""wbs:conn-cartridge-filtration-…"
"""wbs:RO-out-retentate-pressure"""
"""wbs:RO-in-pressure"""
"""wbs:P1-out-pressure"""
"""wbs:RO-out-pressure"""
"""wbs:PXR-brine-out-pressure"""


In [13]:
q = (
    acq.find_all_data()
       .filter_by_unit("KG/s")
)
_ =q.metadata_head()

                Metadata First 10 Rows                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:PXR-brine-out-flow-mass-water                   │
│ wbs:RO-in-flow-mass-water                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-water │
│ wbs:RO-out-flow-mass-tds                            │
│ wbs:RO-out-retentate-flow-mass-tds                  │
│ wbs:RO-in-flow-mass-tds                             │
│ wbs:RO-out-flow-mass-water                          │
│ wbs:RO-out-retentate-flow-mass-water                │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds   │
│ wbs:PXR-brine-out-flow-mass-tds                     │
└─────────────────────────────────────────────────────┘

In [14]:
q = acq.explore().measurement().where(unit="kg/s")
q.metadata()

data
str
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-in-flow-mass-water"""
"""wbs:RO-out-flow-mass-water"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:PXR-brine-out-flow-mass-td…"
"""wbs:RO-out-flow-mass-tds"""
"""wbs:conn-cartridge-filtration-…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-wa…"


In [15]:
q = (
    acq.find_all_data()
        .filter_by_substance("constituent Salt")
        .filter_by_unit("KG/s")
)
_ =q.metadata_head()

               Metadata First 10 Rows                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:RO-out-flow-mass-tds                          │
│ wbs:RO-out-retentate-flow-mass-tds                │
│ wbs:RO-in-flow-mass-tds                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:PXR-brine-out-flow-mass-tds                   │
└───────────────────────────────────────────────────┘

In [16]:
q = acq.explore().measurement().where(unit="kg/s",substance="constituent salt")
q.metadata()

data
str
"""wbs:RO-in-flow-mass-tds"""
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-td…"


The built-in filters map to fixed predicates (`qudt:hasUnit`, `s223:ofSubstance`, `qudt:hasQuantityKind`, `s223:hasMedium`). For any other predicate use `filter_data_nodes` directly (values are exact URIs, no text resolution):

In [17]:
S223 = "http://data.ashrae.org/standard223#"
q_brine = acq.find_all_data().filter_data_nodes(
    predicate=S223 + "ofMedium", value=["urn:nawi-water-ontology#Water-Brine"])
_ = q_brine.metadata_head()

        Metadata First 10 Rows         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:RO-out-retentate-flow-mass-tds  │
│ wbs:PXR-brine-out-tds-concentration │
│ wbs:PXR-brine-out-flow-mass-tds     │
└─────────────────────────────────────┘

## Inspect the query
Every query compiles to SPARQL against the server's graph — nothing is hidden. `show_query_graph` prints the node/edge structure, `to_sparql` returns the compiled query (check it when a result surprises you), `metadata` returns the full result as a polars DataFrame (`include_internals=True` keeps the ref/unit columns).

In [18]:
q = (
    acq.find_all_data()
        .filter_by_substance("constituent Salt")
        .filter_by_unit("KG/s")
)
q.show_query_graph()

QUERY GRAPH

Nodes:
  0 [0] [DATA]  class=*

Edges:

Data nodes:
  0 [0]  filters={http://data.ashrae.org/standard223#ofSubstance=['urn:nawi-water-ontology#Constituent-Salt'], http://qudt.org/schema/qudt/hasUnit=['http://qudt.org/vocab/unit/KiloGM-PER-SEC']}}

Current pointer: 0



In [19]:
print(q.to_sparql())

SELECT DISTINCT ?v0 ?ext0 ?unit0 ?extunit0
WHERE {
  ?v0 <https://brickschema.org/schema/Brick/ref#hasExternalReference> ?ext0 .
  OPTIONAL { ?v0 <http://qudt.org/schema/qudt/hasUnit> ?unit0 . }
  OPTIONAL { ?ext0 <http://qudt.org/schema/qudt/hasUnit> ?extunit0 . }
  { { ?v0 <http://data.ashrae.org/standard223#ofSubstance> <urn:nawi-water-ontology#Constituent-Salt> . } }
  { { ?v0 <http://qudt.org/schema/qudt/hasUnit> <http://qudt.org/vocab/unit/KiloGM-PER-SEC> . } }
}


In [20]:
df_meta = q.metadata()
df_meta

0
str
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-td…"
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-in-flow-mass-tds"""


## Pull timeseries
`dataframe` returns a polars frame. `shape="wide"` puts each data node in its own column, `"narrow"` is long-form. `latest_data` is a shortcut for the most recent point.

In [21]:
q = acq.explore().measurement().where(unit="kg/s",substance="constituent salt")


end = datetime.now(tz=timezone.utc)
start = end - timedelta(hours=10)

df = q.dataframe(start=start, end=end, shape="wide", cast_value="float")
df.head()

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,11.636159,11.606711,11.636159,0.029447,11.606711
2026-08-04 20:53:20.200666 UTC,11.128819,11.099154,11.128819,0.029665,11.099154
2026-08-04 20:53:39.151741 UTC,11.319533,11.289939,11.319533,0.029594,11.289939
2026-08-04 20:54:03.991986 UTC,11.335039,11.30535,11.335039,0.029689,11.30535
2026-08-04 20:54:19.432224 UTC,11.687578,11.658163,11.687578,0.029415,11.658163


In [22]:
q.dataframe(limit=1, order='desc', shape="wide")

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-05 00:04:20.086642 UTC,10.675185,10.645048,10.675185,0.030136,10.645048


## Structured access via DataObject
`data()` returns an object keyed by alias for quick lookups.

In [23]:
import polars as pl


data = q.data(start=start, end=end, cast_value="float")
data.dataframe()

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,11.636159,11.606711,11.636159,0.029447,11.606711
2026-08-04 20:53:20.200666 UTC,11.128819,11.099154,11.128819,0.029665,11.099154
2026-08-04 20:53:39.151741 UTC,11.319533,11.289939,11.319533,0.029594,11.289939
2026-08-04 20:54:03.991986 UTC,11.335039,11.30535,11.335039,0.029689,11.30535
2026-08-04 20:54:19.432224 UTC,11.687578,11.658163,11.687578,0.029415,11.658163
…,…,…,…,…,…
2026-08-05 00:03:20.418760 UTC,10.80747,10.777669,10.80747,0.029801,10.777669
2026-08-05 00:03:35.501514 UTC,10.571054,10.540776,10.571054,0.030278,10.540776
2026-08-05 00:03:49.888609 UTC,10.815503,10.785598,10.815503,0.029905,10.785598


## Inspect units
`units()` returns the effective QUDT unit URI per data alias.

In [24]:
data.units()

{'data': 'http://qudt.org/vocab/unit/KiloGM-PER-SEC'}

## Convert units
`convert_to(target)` accepts any QUDT-recognized identifier (URI, label, symbol, UCUM code). The returned DataObject has values converted and `units()` updated.

In [25]:
data = data.convert_to("kg/min")
data.dataframe().head()

time,data__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,data__wbs:PXR-brine-out-flow-mass-tds,data__wbs:RO-in-flow-mass-tds,data__wbs:RO-out-flow-mass-tds,data__wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,698.16953,696.402685,698.16953,1.766845,696.402685
2026-08-04 20:53:20.200666 UTC,667.72916,665.949253,667.72916,1.779907,665.949253
2026-08-04 20:53:39.151741 UTC,679.171988,677.396365,679.171988,1.775623,677.396365
2026-08-04 20:54:03.991986 UTC,680.102332,678.321004,680.102332,1.781327,678.321004
2026-08-04 20:54:19.432224 UTC,701.254703,699.48978,701.254703,1.764924,699.48978


### Systems

Systems are logical groupings of equipment and junctions (and other systems) in S223 ontology (parent ontology of WaTr)

The systems in the model are:

In [26]:
def list_systems():
    q = acq.find_entity(_class = "System", alias = "Systems")
    return q.metadata()
list_systems()

Systems
str
"""wbs:desalination-system"""
"""wbs:pretreatment-system"""
"""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant"""


In [27]:
q = acq.explore().entity(cls="system")
q.metadata()

system
str
"""wbs:posttreatment-system"""
"""wbs:desalination-system"""
"""wbs:pretreatment-system"""
"""wbs:seawater-ro-plant"""


The systems are hierarchically organized as:

In [28]:
def list_systems_hier():
    q = acq.find_entity(_class = "System", alias = "Systems")
    q = q.find_related(_class = "System", predicates = ['hasMember'], alias = "Subsystem")
    q.metadata_head()
list_systems_hier()

               Metadata First 10 Rows               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Systems               ┃ Subsystem                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:seawater-ro-plant │ wbs:posttreatment-system │
│ wbs:seawater-ro-plant │ wbs:desalination-system  │
│ wbs:seawater-ro-plant │ wbs:pretreatment-system  │
└───────────────────────┴──────────────────────────┘

In [29]:
q = acq.explore().entity(cls="system").related("system").alias("subsystem")
q.metadata()

system,subsystem
str,str
"""wbs:seawater-ro-plant""","""wbs:desalination-system"""
"""wbs:seawater-ro-plant""","""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant""","""wbs:pretreatment-system"""


We can see how many equipment we have in each system:

In [30]:
def list_equipment_by_system(hops = 1):
        q = acq.find_entity(_class = "System", alias = "Systems")
        q = q.find_related(_class = "Equipment", predicates = ['hasMember'], alias = "Equipment", hops = hops, multi_hop_predicates = True)
        q_df = q.metadata()
        return q_df.group_by("Systems").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count", descending=True)

list_equipment_by_system()

Systems,equipment_count
str,u32
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


In [31]:
q = acq.explore().entity(cls="System").related("Equipment")
q.metadata()

System,Equipment
str,str
"""wbs:desalination-system""","""wbs:RO"""
"""wbs:pretreatment-system""","""wbs:ferric-chloride-addition"""
"""wbs:seawater-ro-plant""","""wbs:P1"""
"""wbs:pretreatment-system""","""wbs:static-mixer"""
"""wbs:seawater-ro-plant""","""wbs:backwash-handling"""
…,…
"""wbs:seawater-ro-plant""","""wbs:media-filtration"""
"""wbs:seawater-ro-plant""","""wbs:uv-aop"""
"""wbs:posttreatment-system""","""wbs:co2-addition"""


In [32]:
q.metadata().group_by("System").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count",descending=True)

System,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


These are the number of equipment directly a member of these systems.

If we increase the hops, you'll see total number equipments in each system

In [33]:
list_equipment_by_system(3)

Systems,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


In [34]:
q = acq.explore().entity(cls="System").related("Equipment",via="has member",nearest=False)
q.metadata().group_by("System").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count",descending=True)

System,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


Let's find the pumps in a specific system:


In [35]:
def list_equipment_in_system(system, equipment):
    q = acq.find_entity(uri=system, alias = "system").find_related(_class=equipment, alias = "equipment", hops=1)
    q.metadata_head()

list_equipment_in_system('wbs:pretreatment-system', 'pump')

         Metadata First 10 Rows         
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ system                  ┃ equipment  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ wbs:pretreatment-system │ wbs:intake │
└─────────────────────────┴────────────┘

In [36]:
q= acq.explore().entity(uri='wbs:pretreatment-system').alias("system").related("pump")
q.metadata()

system,pump
str,str
"""wbs:pretreatment-system""","""wbs:intake"""


Let's find all the pumps and their data

In [37]:
def all_pumps_and_their_data():
    q = acq.find_entity(_class="pump", alias="pump").find_all_data()
    q.metadata_head()
    return q.data(limit=10).dataframe()

all_pumps_and_their_data().head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-toc-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-mechanical-power         │
│ wbs:P2     │ wbs:P2-mechanical-power         │
│ wbs:P2     │ wbs:P2-efficiency               │
└────────────┴─────────────────────────────────┘

time,pump_data__wbs:intake-in-flow-rate,pump_data__wbs:intake-in-tds-concentration,pump_data__wbs:intake-in-toc-concentration,pump_data__wbs:intake-in-tss-concentration,pump_data__wbs:P1-efficiency,pump_data__wbs:P1-mechanical-power,pump_data__wbs:P1-out-pressure,pump_data__wbs:P2-efficiency,pump_data__wbs:P2-mechanical-power
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-08-04 20:53:02.260671 UTC,0.319508,36.418992,0.004408,0.035273,0.8,1.1117e6,7e6,0.8,133801.231609
2026-08-04 20:53:20.200666 UTC,0.306413,36.319629,0.003465,0.034358,0.8,1.0956e6,7e6,0.8,122461.359676
2026-08-04 20:53:39.151741 UTC,0.311903,36.291785,0.003621,0.023289,0.8,1.1063e6,7e6,0.8,127048.628758
2026-08-04 20:54:03.991986 UTC,0.310713,36.48073,0.00376,0.02101,0.8,1.0995e6,7e6,0.8,126537.96316
2026-08-04 20:54:19.432224 UTC,0.321326,36.372963,0.003746,0.02581,0.8,1.1163e6,7e6,0.8,135316.280988


In [38]:
q = acq.explore().entity("pump").measurement()
q.metadata().head()

pump,pump_data
str,str
"""wbs:intake""","""wbs:intake-in-tds-concentratio…"
"""wbs:P1""","""wbs:P1-efficiency"""
"""wbs:P1""","""wbs:P1-mechanical-power"""
"""wbs:P2""","""wbs:P2-efficiency"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…"


Let's find all the data generating entites within a system:

In [39]:
def find_all_sensors(system):
    q = (acq.find_entity(uri=system, alias="backwash")
         .find_related(_class="equipment", alias="equipment",predicates=['hasMember'], hops=1)
         .find_data(alias = "sensors"))
    q_df = q.metadata(include_internals=True)
    q_df = q_df.drop([pl.col('backwash'),pl.col('sensors_ref'),pl.col('extunit4')])
    return q_df

system = 'wbs:pretreatment-system'
find_all_sensors(system)

equipment,sensors,unit4
str,str,str
"""wbs:intake""","""wbs:intake-in-toc-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:cartridge-filtration""","""wbs:cartridge-filtration-out-t…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""


In [40]:
q = (acq.explore().entity(uri = 'wbs:pretreatment-system').drop()
     .related('equipment').measurement(alias="sensor").include("unit"))
q.metadata()

equipment,sensor,sensor.unit
str,str,str
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:cartridge-filtration""","""wbs:cartridge-filtration-out-t…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-toc-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
